# Understanding the Training Process: Forward and Backward Propagation

This notebook explores the fundamental components of neural network training, focusing on the forward and backward propagation processes that enable neural networks to learn from data.

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set plotting style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)

## 2. Neural Network Basics

Before diving into forward and backward propagation, let's understand the basic components of neural networks:

- **Neurons**: The fundamental units that receive inputs, apply activation functions, and produce outputs
- **Layers**: Collections of neurons (input layer, hidden layers, output layer)
- **Weights and Biases**: Learnable parameters that determine the network's behavior
- **Activation Functions**: Non-linear transformations applied to neuron outputs

A typical neural network architecture looks like this:

In [ ]:
# Visualization of a simple neural network structure
def plot_neural_network():
    fig, ax = plt.subplots()
    
    # Number of neurons in each layer
    n_input = 3
    n_hidden = 4
    n_output = 2
    
    # Positions of neurons
    input_neurons_y = np.linspace(0.2, 0.8, n_input)
    hidden_neurons_y = np.linspace(0.2, 0.8, n_hidden)
    output_neurons_y = np.linspace(0.3, 0.7, n_output)
    
    # Draw neurons
    input_x, hidden_x, output_x = 0.1, 0.5, 0.9
    neuron_radius = 0.02
    
    # Input layer
    input_neurons = []
    for y in input_neurons_y:
        circle = plt.Circle((input_x, y), neuron_radius, fill=True, color='blue', alpha=0.7)
        ax.add_patch(circle)
        input_neurons.append((input_x, y))
        
    # Hidden layer
    hidden_neurons = []
    for y in hidden_neurons_y:
        circle = plt.Circle((hidden_x, y), neuron_radius, fill=True, color='red', alpha=0.7)
        ax.add_patch(circle)
        hidden_neurons.append((hidden_x, y))
    
    # Output layer
    output_neurons = []
    for y in output_neurons_y:
        circle = plt.Circle((output_x, y), neuron_radius, fill=True, color='green', alpha=0.7)
        ax.add_patch(circle)
        output_neurons.append((output_x, y))
    
    # Draw connections (weights)
    # Input to hidden
    for i_pos in input_neurons:
        for h_pos in hidden_neurons:
            plt.plot([i_pos[0], h_pos[0]], [i_pos[1], h_pos[1]], 'k-', alpha=0.3)
    
    # Hidden to output
    for h_pos in hidden_neurons:
        for o_pos in output_neurons:
            plt.plot([h_pos[0], o_pos[0]], [h_pos[1], o_pos[1]], 'k-', alpha=0.3)
    
    # Add layer labels
    plt.text(input_x, 0.95, 'Input Layer', ha='center', fontsize=12)
    plt.text(hidden_x, 0.95, 'Hidden Layer', ha='center', fontsize=12)
    plt.text(output_x, 0.95, 'Output Layer', ha='center', fontsize=12)
    
    # Add annotation showing forward and backward passes
    plt.arrow(0.3, 0.1, 0.4, 0, head_width=0.02, head_length=0.02, fc='blue', ec='blue')
    plt.text(0.5, 0.07, 'Forward Propagation', ha='center', fontsize=11, color='blue')
    
    plt.arrow(0.7, 0.03, -0.4, 0, head_width=0.02, head_length=0.02, fc='red', ec='red')
    plt.text(0.5, 0.0, 'Backward Propagation', ha='center', fontsize=11, color='red')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.title('Neural Network Architecture', fontsize=14)
    plt.show()

plot_neural_network()

## 3. Forward Propagation Explained

Forward propagation is the process of passing input data through the neural network to generate predictions. Let's understand it step by step for a simple 2-layer neural network:

### Mathematical Representation

For a simple 2-layer neural network:

1. **Input Layer to Hidden Layer**:
   - $\mathbf{Z}^{[1]} = \mathbf{W}^{[1]} \mathbf{X} + \mathbf{b}^{[1]}$
   - $\mathbf{A}^{[1]} = g^{[1]}(\mathbf{Z}^{[1]})$ (applying activation function)

2. **Hidden Layer to Output Layer**:
   - $\mathbf{Z}^{[2]} = \mathbf{W}^{[2]} \mathbf{A}^{[1]} + \mathbf{b}^{[2]}$
   - $\mathbf{A}^{[2]} = g^{[2]}(\mathbf{Z}^{[2]})$ (applying output activation function)

Where:
- $\mathbf{W}^{[l]}$ is the weight matrix for layer $l$
- $\mathbf{b}^{[l]}$ is the bias vector for layer $l$
- $\mathbf{Z}^{[l]}$ is the weighted input to layer $l$
- $\mathbf{A}^{[l]}$ is the activation output from layer $l$
- $g^{[l]}$ is the activation function for layer $l$

In [ ]:
# Implement forward propagation from scratch
def initialize_parameters(n_input, n_hidden, n_output):
    """Initialize weights and biases for a 2-layer neural network"""
    np.random.seed(42)
    
    # Xavier/Glorot initialization for better convergence
    W1 = np.random.randn(n_hidden, n_input) * np.sqrt(1/n_input)
    b1 = np.zeros((n_hidden, 1))
    W2 = np.random.randn(n_output, n_hidden) * np.sqrt(1/n_hidden)
    b2 = np.zeros((n_output, 1))
    
    parameters = {
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2
    }
    
    return parameters

def sigmoid(Z):
    """Sigmoid activation function"""
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    """ReLU activation function"""
    return np.maximum(0, Z)

def forward_propagation(X, parameters):
    """Perform forward propagation for a 2-layer neural network"""
    # Retrieve parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # Forward propagation steps
    Z1 = np.dot(W1, X) + b1  # Linear transformation
    A1 = relu(Z1)            # Activation (ReLU)
    Z2 = np.dot(W2, A1) + b2 # Linear transformation
    A2 = sigmoid(Z2)         # Output activation (sigmoid)
    
    # Store values for backward pass
    cache = {
        "Z1": Z1, "A1": A1,
        "Z2": Z2, "A2": A2,
        "X": X
    }
    
    return A2, cache

# Example of forward propagation
n_input, n_hidden, n_output = 2, 4, 1
X_sample = np.random.randn(n_input, 5)  # 5 samples with 2 features each
parameters = initialize_parameters(n_input, n_hidden, n_output)

# Perform forward propagation
output, cache = forward_propagation(X_sample, parameters)

print(f"Input shape: {X_sample.shape}")
print(f"Output shape: {output.shape}")
print(f"Output predictions:\n{output}")

## 4. Loss Function Calculation

After forward propagation, we need to quantify how well our network is performing by comparing the predictions to the actual target values. This is done using a loss function.

Common loss functions include:
- **Binary Cross-Entropy**: For binary classification
- **Categorical Cross-Entropy**: For multi-class classification
- **Mean Squared Error (MSE)**: For regression tasks

In [ ]:
def compute_cost(A2, Y):
    """Compute binary cross-entropy loss"""
    m = Y.shape[1]  # Number of examples
    
    # Binary cross-entropy loss
    # L = -1/m * Σ [Y * log(A2) + (1-Y) * log(1-A2)]
    logprobs = np.multiply(np.log(A2), Y) + np.multiply(np.log(1 - A2), 1 - Y)
    cost = -1/m * np.sum(logprobs)
    
    # Ensure cost is a scalar
    cost = float(np.squeeze(cost))
    
    return cost

# Generate random target values for our example
Y_sample = np.random.randint(0, 2, (1, X_sample.shape[1]))

# Calculate loss
loss = compute_cost(output, Y_sample)
print(f"Loss: {loss:.6f}")

## 5. Backward Propagation Explained

Backward propagation (backpropagation) is the process of calculating gradients of the loss function with respect to the weights and biases of the neural network. These gradients indicate the direction to adjust the parameters to minimize the loss.

### The Mathematics of Backpropagation

We use the chain rule of calculus to compute derivatives of the loss with respect to each parameter:

1. **Output Layer**:
   - $\frac{\partial L}{\partial Z^{[2]}} = A^{[2]} - Y$ (for binary cross-entropy with sigmoid)
   - $\frac{\partial L}{\partial W^{[2]}} = \frac{1}{m} \frac{\partial L}{\partial Z^{[2]}} A^{[1]T}$
   - $\frac{\partial L}{\partial b^{[2]}} = \frac{1}{m} \sum \frac{\partial L}{\partial Z^{[2]}}$

2. **Hidden Layer**:
   - $\frac{\partial L}{\partial A^{[1]}} = W^{[2]T} \frac{\partial L}{\partial Z^{[2]}}$
   - $\frac{\partial L}{\partial Z^{[1]}} = \frac{\partial L}{\partial A^{[1]}} * g'^{[1]}(Z^{[1]})$ (element-wise multiplication with activation derivative)
   - $\frac{\partial L}{\partial W^{[1]}} = \frac{1}{m} \frac{\partial L}{\partial Z^{[1]}} X^T$
   - $\frac{\partial L}{\partial b^{[1]}} = \frac{1}{m} \sum \frac{\partial L}{\partial Z^{[1]}}$

In [ ]:
def relu_derivative(Z):
    """Derivative of the ReLU function"""
    return np.where(Z > 0, 1, 0)

def backward_propagation(parameters, cache, Y):
    """Perform backward propagation to compute gradients"""
    m = Y.shape[1]  # Number of examples
    
    # Retrieve parameters
    W1 = parameters["W1"]
    W2 = parameters["W2"]
    
    # Retrieve from cache
    A1 = cache["A1"]
    A2 = cache["A2"]
    Z1 = cache["Z1"]
    X = cache["X"]
    
    # Backward propagation
    # Output layer
    dZ2 = A2 - Y  # Derivative of binary cross-entropy with sigmoid
    dW2 = 1/m * np.dot(dZ2, A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
    
    # Hidden layer
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = np.multiply(dA1, relu_derivative(Z1))  # Element-wise multiplication
    dW1 = 1/m * np.dot(dZ1, X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)
    
    # Store gradients
    gradients = {
        "dW1": dW1, "db1": db1,
        "dW2": dW2, "db2": db2
    }
    
    return gradients

# Compute gradients
gradients = backward_propagation(parameters, cache, Y_sample)

# Print gradient shapes
for key, value in gradients.items():
    print(f"{key} shape: {value.shape}, mean: {np.mean(value):.6f}")

## 6. Implementing a Simple Neural Network from Scratch

Now let's put together the forward and backward propagation to create a complete neural network training process:

In [ ]:
def update_parameters(parameters, gradients, learning_rate):
    """Update parameters using gradient descent"""
    # Retrieve parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    # Retrieve gradients
    dW1 = gradients["dW1"]
    db1 = gradients["db1"]
    dW2 = gradients["dW2"]
    db2 = gradients["db2"]
    
    # Update parameters using gradient descent
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    
    # Store updated parameters
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    
    return parameters

def predict(X, parameters):
    """Make predictions using trained parameters"""
    A2, _ = forward_propagation(X, parameters)
    predictions = (A2 > 0.5).astype(int)
    return predictions

def train_neural_network(X, Y, hidden_size=4, learning_rate=0.1, num_iterations=10000, print_cost=True):
    """Train a 2-layer neural network"""
    np.random.seed(42)
    n_input = X.shape[0]
    n_output = Y.shape[0]
    costs = []
    
    # Initialize parameters
    parameters = initialize_parameters(n_input, hidden_size, n_output)
    
    # Training loop
    for i in range(num_iterations):
        # Forward propagation
        A2, cache = forward_propagation(X, parameters)
        
        # Compute cost
        cost = compute_cost(A2, Y)
        
        # Backward propagation
        gradients = backward_propagation(parameters, cache, Y)
        
        # Update parameters
        parameters = update_parameters(parameters, gradients, learning_rate)
        
        # Print and store cost
        if print_cost and i % 1000 == 0:
            print(f"Cost after iteration {i}: {cost:.6f}")
        if i % 100 == 0:
            costs.append(cost)
    
    return parameters, costs

## 7. Visualizing Gradients and Learning

Let's create a simple binary classification problem and visualize how our neural network learns:

In [ ]:
# Generate a moon-shaped dataset for binary classification
X_data, y_data = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).T  # Shape: (2, 800)
X_test = scaler.transform(X_test).T        # Shape: (2, 200)
Y_train = y_train.reshape(1, -1)           # Shape: (1, 800)
Y_test = y_test.reshape(1, -1)             # Shape: (1, 200)

# Visualize the dataset
plt.figure(figsize=(10, 6))
plt.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='viridis', edgecolors='k', s=50, alpha=0.7)
plt.title("Moon-shaped binary classification dataset", fontsize=14)
plt.xlabel("Feature 1", fontsize=12)
plt.ylabel("Feature 2", fontsize=12)
plt.colorbar(label="Class")
plt.show()

# Train our neural network
parameters, costs = train_neural_network(X_train, Y_train, hidden_size=5, learning_rate=0.1, num_iterations=10000)

# Plot the cost over iterations
plt.figure(figsize=(10, 6))
plt.plot(costs)
plt.xlabel("Iterations (hundreds)", fontsize=12)
plt.ylabel("Cost", fontsize=12)
plt.title("Learning Curve: Cost vs Iterations", fontsize=14)
plt.grid(True)
plt.show()

# Evaluate the model
train_predictions = predict(X_train, parameters)
test_predictions = predict(X_test, parameters)

train_accuracy = np.mean(train_predictions == Y_train) * 100
test_accuracy = np.mean(test_predictions == Y_test) * 100

print(f"Train accuracy: {train_accuracy:.2f}%")
print(f"Test accuracy: {test_accuracy:.2f}%")

### Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(X, y, parameters):
    """Visualize the decision boundary created by the neural network"""
    # Set min and max values and give it some padding
    x_min, x_max = X[0, :].min() - 1, X[0, :].max() + 1
    y_min, y_max = X[1, :].min() - 1, X[1, :].max() + 1
    h = 0.01
    
    # Generate a grid of points
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predict the function value for the grid
    Z = predict(np.c_[xx.ravel(), yy.ravel()].T, parameters)
    Z = Z.reshape(xx.shape)
    
    # Plot the contour and training examples
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    plt.scatter(X[0, :], X[1, :], c=y.squeeze(), cmap=plt.cm.Spectral, edgecolors='k', s=40)
    plt.xlabel('Feature 1', fontsize=12)
    plt.ylabel('Feature 2', fontsize=12)
    plt.title('Decision Boundary', fontsize=14)
    plt.show()

# Plot decision boundary
plot_decision_boundary(X_train, Y_train, parameters)

## 8. Practical Example with PyTorch

Let's see how forward and backward propagation are implemented in modern deep learning frameworks like PyTorch:

In [ ]:
# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train.T, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.T, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.T, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.T, dtype=torch.float32)

# Define a neural network model
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNN, self).__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # Forward pass (automatically stores computational graph for backpropagation)
        z1 = self.layer1(x)
        a1 = self.relu(z1)
        z2 = self.layer2(a1)
        a2 = self.sigmoid(z2)
        return a2

# Initialize model, loss function, and optimizer
input_size = 2
hidden_size = 5
output_size = 1
model = SimpleNN(input_size, hidden_size, output_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
losses = []

for epoch in range(epochs):
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, Y_train_tensor)
    
    # Backward pass and optimization
    optimizer.zero_grad()   # Clear previous gradients
    loss.backward()         # Compute gradients through backpropagation
    optimizer.step()        # Update parameters
    
    if (epoch+1) % 100 == 0:
        losses.append(loss.item())
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Plot training progress
plt.figure(figsize=(10, 6))
plt.plot(range(100, epochs+1, 100), losses)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss (PyTorch)', fontsize=14)
plt.grid(True)
plt.show()

# Evaluate the model
with torch.no_grad():
    train_outputs = model(X_train_tensor)
    train_predictions = (train_outputs > 0.5).float()
    train_accuracy = (train_predictions == Y_train_tensor).float().mean() * 100
    
    test_outputs = model(X_test_tensor)
    test_predictions = (test_outputs > 0.5).float()
    test_accuracy = (test_predictions == Y_test_tensor).float().mean() * 100

print(f"Train accuracy (PyTorch): {train_accuracy.item():.2f}%")
print(f"Test accuracy (PyTorch): {test_accuracy.item():.2f}%")

## 9. Gradient Descent Optimization

Gradient descent is the optimization algorithm used to minimize the loss function by iteratively adjusting the model parameters.

There are three main variants of gradient descent:

1. **Batch Gradient Descent**: Uses the entire training dataset to compute gradients in each iteration.
2. **Stochastic Gradient Descent (SGD)**: Uses a single random training example to compute gradients in each iteration.
3. **Mini-batch Gradient Descent**: Uses a small random batch of training examples (e.g., 32, 64, 128) to compute gradients in each iteration.

Let's visualize the gradient descent optimization process:

In [ ]:
def plot_gradient_descent_2d():
    """Visualize gradient descent optimization in 2D"""
    # Create a simple loss function: f(x, y) = x^2 + 2y^2
    def loss_function(x, y):
        return x**2 + 2*y**2
    
    def gradient(x, y):
        return np.array([2*x, 4*y])
    
    # Create a grid of values
    x = np.linspace(-4, 4, 100)
    y = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z = loss_function(X, Y)
    
    # Initialize starting point and parameters
    learning_rate = 0.1
    num_iterations = 20
    x_current, y_current = -3.5, 2.5
    path_x, path_y = [x_current], [y_current]
    
    # Perform gradient descent
    for _ in range(num_iterations):
        grad = gradient(x_current, y_current)
        x_current -= learning_rate * grad[0]
        y_current -= learning_rate * grad[1]
        path_x.append(x_current)
        path_y.append(y_current)
    
    # Plotting
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot the contour
    contour = ax.contourf(X, Y, Z, 50, cmap='viridis', alpha=0.8)
    ax.contour(X, Y, Z, 20, colors='black', alpha=0.4, linestyles='solid')
    plt.colorbar(contour, ax=ax, label='Loss')
    
    # Plot the path
    ax.plot(path_x, path_y, 'ro-', linewidth=2, markersize=8, alpha=0.7)
    ax.plot(0, 0, 'g*', markersize=15)  # Global minimum
    
    # Add annotations
    ax.annotate('Start', xy=(path_x[0], path_y[0]), xytext=(path_x[0]-0.5, path_y[0]+0.5),
                arrowprops=dict(arrowstyle='->'), fontsize=12)
    ax.annotate('Minimum', xy=(0, 0), xytext=(0.5, 0.5),
                arrowprops=dict(arrowstyle='->'), fontsize=12)
    
    ax.set_xlabel('Parameter 1', fontsize=12)
    ax.set_ylabel('Parameter 2', fontsize=12)
    ax.set_title('Gradient Descent Optimization', fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

plot_gradient_descent_2d()

## 10. Common Challenges in Training Neural Networks

Training neural networks can be challenging due to several issues:

### 1. Vanishing/Exploding Gradients

- **Vanishing Gradients**: Gradients become extremely small, making learning very slow or stopping entirely
- **Exploding Gradients**: Gradients become extremely large, causing unstable updates

**Solutions**:
- Proper weight initialization (He, Xavier/Glorot)
- Using activation functions that help mitigate gradient issues (ReLU, Leaky ReLU)
- Batch normalization
- Gradient clipping

### 2. Choosing Appropriate Learning Rates

- Too small: Training is very slow
- Too large: Training becomes unstable or diverges

**Solutions**:
- Learning rate schedules
- Adaptive learning rate optimizers (Adam, RMSprop)
- Learning rate warmup

### 3. Local Minima and Saddle Points

- Neural networks have non-convex loss landscapes with many local minima and saddle points

**Solutions**:
- Momentum-based optimizers
- Stochastic methods like SGD with noise
- Ensembling multiple models

### 4. Overfitting

- Model performs well on training data but poorly on unseen data

**Solutions**:
- Regularization (L1, L2)
- Dropout
- Early stopping
- Data augmentation
- Batch normalization

## Summary

In this notebook, we covered the essential components of neural network training, focusing on forward and backward propagation:

1. **Forward Propagation**:
   - Process of passing input data through the network to generate predictions
   - Sequential application of linear transformations and non-linear activations

2. **Loss Calculation**:
   - Quantifying the difference between predictions and actual targets
   - Different loss functions for different tasks (regression, classification)

3. **Backward Propagation**:
   - Computing gradients of the loss with respect to model parameters
   - Using the chain rule of calculus to propagate errors backward

4. **Parameter Updates**:
   - Using gradients to adjust weights and biases
   - Various optimization algorithms to improve convergence

Understanding these processes is fundamental to developing, training, and debugging neural networks effectively.